# SiPM-Specific Parameter Time Stability & Variance Pooling Analysis

This notebook quantifies the **intrinsic temporal stability** of SND@LHC timing calibration parameters:
1. **Model Formulation:** Each SiPM channel $ has an intrinsic, time-invariant parameter value $	heta_i$ with run-to-run measurement variance $\sigma_i^2 = \mathrm{Var}_r(p_{i,r})$.
2. **Channel-Level Variance Extraction:** For each channel, we evaluate the run-to-run dispersion:
   - Robust standard deviation: $\hat{\sigma}_i = rac{\mathrm{IQR}_r(p_{i,r})}{1.349}$
   - Normalized dispersion (Coefficient of Variation): $\mathrm{CV}_i = rac{\hat{\sigma}_i}{|\hat{	heta}_i|}$
3. **Unified Global Stability Metric (Variance Pooling):**
   - Pooled Root-Mean-Variance across all $ detector channels:
     995845ar{\sigma}_{\mathrm{pool}} = \sqrt{rac{1}{M} \sum_{i=1}^M \sigma_i^2}, \qquad ar{\sigma}_{\mathrm{med}} = \mathrm{median}_i(\hat{\sigma}_i)995845
   - Evaluated both **within eras (Intra-Year Stability)** and **across all eras (Inter-Year Stability)**.


In [ ]:
import os
import glob
import json
import concurrent.futures
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats

try:
    import ddfUtils.mpl_styling as dms
except ImportError:
    pass

plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["font.size"] = 11

fits_dir = "/eos/user/i/idioniso/Timing/timewalk_tw/fits"
json_files = sorted(glob.glob(os.path.join(fits_dir, "run*_timing_calibration.json")))

def get_run_period(run: int) -> str:
    if 5000 <= run < 6000:
        return "Cat II (2022)"
    elif 6000 <= run < 7000:
        return "Cat III (2023)"
    elif 8000 <= run < 10000:
        return "Cat III (2024)"
    elif 10000 <= run < 13000:
        return "Cat III (2025)"
    elif 13000 <= run < 14000:
        return "Cat III (2026)"
    return "Other"

print(f"Found {len(json_files)} calibrated run files in {fits_dir}")


In [ ]:
def parse_file(jf):
    with open(jf, "r") as f:
        data = json.load(f)
    r = data.get("run", 0)
    period = get_run_period(r)
    tw_ds = data.get("timewalk", {}).get("DS", {})
    al_ds = data.get("alignment", {}).get("DS", {})
    
    records = []
    for ch_key, tw_p in tw_ds.items():
        det_id, sipm = [int(p) for p in ch_key.split("_")]
        subsys = det_id // 10000
        plane = (det_id % 10000) // 1000
        bar = det_id % 1000
        side = "left" if sipm < 8 else "right"
        system_name = "Veto" if subsys == 1 else ("US" if subsys == 2 else "DS")
        
        d_val = al_ds.get(ch_key, [np.nan])[0]
        alpha = tw_p.get("alpha", np.nan)
        beta = tw_p.get("beta", np.nan)
        qdc0 = tw_p.get("qdc0", np.nan)
        gamma = tw_p.get("gamma", np.nan)
        sigma_sys = tw_p.get("sigma_sys", np.nan)
        chi2_tot = tw_p.get("chi2_tot_ndf", np.nan)
        
        # Good-fit quality check
        is_good = (
            (chi2_tot is not None and np.isfinite(chi2_tot) and chi2_tot <= 2.0) and
            (sigma_sys is not None and np.isfinite(sigma_sys) and 0.0 < sigma_sys <= 0.15) and
            (alpha is not None and -80.0 <= alpha <= -5.0) and
            (beta is not None and 0.2 <= beta <= 4.0) and
            (qdc0 is not None and -15.0 <= qdc0 <= -2.0)
        )
        
        records.append({
            "run": r, "period": period, "channel": ch_key, "detID": det_id,
            "subsys": subsys, "system": system_name, "plane": plane, "bar": bar,
            "sipm": sipm, "side": side, "d_sipm": d_val, "alpha": alpha,
            "beta": beta, "qdc0": qdc0, "gamma": gamma, "sigma_sys": sigma_sys,
            "chi2_tot": chi2_tot, "is_good": is_good
        })
    return records

with concurrent.futures.ThreadPoolExecutor(max_workers=8) as ex:
    results = list(ex.map(parse_file, json_files))

all_ch = [rec for r in results for rec in r]
df_all = pd.DataFrame(all_ch)
df_good = df_all[df_all["is_good"]].copy()

print(f"Ingested {len(df_all):,} total records ({len(df_good):,} good-fit records, {len(df_good)/len(df_all)*100:.1f}%).")
print(f"Runs per period in good-fit dataset:")
for p, count in df_good.groupby("period")["run"].nunique().items():
    print(f"  {p:15s}: {count} runs")


---
## 1. Per-Channel Variance & Unified Pooled Stability Metrics

For each channel $, we evaluate:
- **Sample Median:** $\hat{	heta}_i = \mathrm{median}_r(p_{i,r})$
- **Robust Channel Standard Deviation:** $\hat{\sigma}_i = rac{\mathrm{IQR}_r(p_{i,r})}{1.349}$
- **Channel Variance:** $\hat{\sigma}_i^2$
- **Relative Dispersion:** $\mathrm{CV}_i = rac{\hat{\sigma}_i}{|\hat{	heta}_i|} 	imes 100\%$

Then, we aggregate over all channels in the detector to obtain:
- **Unified Median Repeatability:** $\mathrm{Med}(\sigma) = \mathrm{median}_i(\hat{\sigma}_i)$
- **Unified Pooled Standard Deviation:** $\sigma_{\mathrm{pool}} = \sqrt{rac{1}{M} \sum_{i=1}^M \hat{\sigma}_i^2}$


In [ ]:
def compute_channel_stability(df_input, min_runs=6):
    records = []
    params = ["d_sipm", "alpha", "beta", "qdc0", "sigma_sys"]
    
    for ch_key, group in df_input.groupby("channel"):
        if len(group) < min_runs:
            continue
            
        rec = {
            "channel": ch_key,
            "system": group["system"].iloc[0],
            "plane": group["plane"].iloc[0],
            "bar": group["bar"].iloc[0],
            "sipm": group["sipm"].iloc[0],
            "side": group["side"].iloc[0],
            "n_runs": len(group)
        }
        
        for p in params:
            vals = group[p].dropna().values
            if len(vals) < min_runs:
                continue
            med = float(np.median(vals))
            iqr = float(np.percentile(vals, 75) - np.percentile(vals, 25))
            rob_std = float(iqr / 1.349)
            rob_var = float(rob_std ** 2)
            cv = float(rob_std / abs(med) * 100.0) if abs(med) > 1e-4 else np.nan
            
            rec[f"{p}_med"] = med
            rec[f"{p}_std"] = rob_std
            rec[f"{p}_var"] = rob_var
            rec[f"{p}_cv"] = cv
            
        records.append(rec)
    return pd.DataFrame(records)

# 1. Global stability across all runs and years
df_stab_global = compute_channel_stability(df_good, min_runs=8)

# 2. Intra-year stability within Cat II (2022, 33 runs) to isolate pure electronic stability from clock shifts
df_stab_2022 = compute_channel_stability(df_good[df_good["period"] == "Cat II (2022)"], min_runs=8)

print(f"Evaluated global channel stability for {len(df_stab_global):,} SiPM channels.")
print(f"Evaluated 2022 intra-year stability for {len(df_stab_2022):,} SiPM channels.")


In [ ]:
def summarize_pooled_stability(df_stab, title=""):
    params_info = [
        ("d_sipm",    "d_SiPM [ns]",             1.0,    "ns"),
        ("alpha",     "Curvature (alpha)",       1.0,    ""),
        ("beta",      "Scale Factor (beta)",     1.0,    ""),
        ("qdc0",      "Threshold Q0 [QDC]",      1.0,    "QDC"),
        ("sigma_sys", "Sys Error Floor [ps]",   1000.0, "ps")
    ]
    
    rows = []
    for col_key, name, scale, unit in params_info:
        stds = df_stab[f"{col_key}_std"].dropna().values * scale
        vars_ = df_stab[f"{col_key}_var"].dropna().values * (scale ** 2)
        cvs = df_stab[f"{col_key}_cv"].dropna().values
        
        pooled_std = float(np.sqrt(np.mean(vars_)))
        med_std = float(np.median(stds))
        p75_std = float(np.percentile(stds, 75))
        med_cv = float(np.median(cvs))
        
        unit_str = f" {unit}" if unit else ""
        rows.append({
            "Parameter": name,
            "Pooled Std (Root-Mean-Var)": f"{pooled_std:.3f}{unit_str}",
            "Median Std (Repeatability)": f"{med_std:.3f}{unit_str}",
            "P75 Std": f"{p75_std:.3f}{unit_str}",
            "Median Relative CV": f"{med_cv:.2f}%"
        })
    df_res = pd.DataFrame(rows)
    print(f"=== Unified Time Stability Metrics: {title} ===")
    print(df_res.to_string(index=False))
    return df_res

table_global = summarize_pooled_stability(df_stab_global, "Global (All Runs & Eras Combined)")
print("")
table_2022 = summarize_pooled_stability(df_stab_2022, "Intra-Year (Pure Run-by-Run Stability within 2022)")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9.5))
axes_flat = axes.flatten()

plot_cfgs = [
    ("d_sipm",    r"Fine Time Alignment ($d_{\mathrm{SiPM}}$)",   "ps",  1000.0, (0, 150),   "#1f77b4"),
    ("alpha",     r"Curvature Scale ($\alpha$)",                  "",    1.0,    (0, 10),    "#ff7f0e"),
    ("beta",      r"Scale Factor ($\beta$)",                     "",    1.0,    (0, 0.8),   "#2ca02c"),
    ("qdc0",      r"Threshold Offset ($q_0$)",                   "QDC", 1.0,    (0, 1.5),   "#d62728"),
    ("sigma_sys", r"Systematic Error Floor ($\sigma_{\mathrm{sys}}$)", "ps", 1000.0, (0, 30),    "#9467bd"),
]

for ax, (p_col, title, unit, scale, xlim, col) in zip(axes_flat[:5], plot_cfgs):
    # Intra-year channel std
    intra_stds = df_stab_2022[f"{p_col}_std"].dropna().values * scale
    intra_vars = df_stab_2022[f"{p_col}_var"].dropna().values * (scale ** 2)
    pool_intra = np.sqrt(np.mean(intra_vars))
    med_intra = np.median(intra_stds)
    
    # Global channel std (all years)
    glob_stds = df_stab_global[f"{p_col}_std"].dropna().values * scale
    glob_vars = df_stab_global[f"{p_col}_var"].dropna().values * (scale ** 2)
    pool_glob = np.sqrt(np.mean(glob_vars))
    med_glob = np.median(glob_stds)
    
    bins = np.linspace(xlim[0], xlim[1], 45)
    ax.hist(intra_stds, bins=bins, density=True, histtype="step", linewidth=2.0, color="#1f77b4",
            label=f"Intra-Year (2022): Med={med_intra:.2f}, Pooled={pool_intra:.2f}")
    ax.hist(glob_stds, bins=bins, density=True, histtype="step", linewidth=2.0, color="#ff7f0e", linestyle="--",
            label=f"All Eras Combined: Med={med_glob:.2f}, Pooled={pool_glob:.2f}")
    
    ax.axvline(med_intra, color="#1f77b4", linestyle=":", linewidth=1.5)
    ax.axvline(med_glob, color="#ff7f0e", linestyle=":", linewidth=1.5)
    
    unit_label = f" [{unit}]" if unit else ""
    ax.set_xlabel(r"Per-Channel Standard Deviation $\hat{\sigma}_i$", fontsize=11)
    ax.set_ylabel("Probability Density", fontsize=11)
    ax.set_title(title, fontsize=12, fontweight="bold", pad=6)
    ax.set_xlim(xlim)
    ax.grid(True, linestyle=":", alpha=0.5)
    ax.legend(loc="upper right", fontsize=9.0, framealpha=0.92)

# Panel 6: Summary of Relative Dispersion CV (%) across all parameters
ax_cv = axes_flat[5]
bins_cv = np.linspace(0, 30, 50)
for p_col, name, col in [("d_sipm", r"$d_{\mathrm{SiPM}}$", "#1f77b4"),
                         ("alpha", r"$\alpha$", "#ff7f0e"),
                         ("beta", r"$\beta$", "#2ca02c"),
                         ("qdc0", r"$q_0$", "#d62728"),
                         ("sigma_sys", r"$\sigma_{\mathrm{sys}}$", "#9467bd")]:
    cv_vals = df_stab_2022[f"{p_col}_cv"].dropna().values
    ax_cv.hist(cv_vals, bins=bins_cv, density=True, histtype="step", linewidth=2.0, color=col,
               label=f"{name} (Med CV = {np.median(cv_vals):.1f}%)")

ax_cv.set_xlabel("Relative Coefficient of Variation ($) [%]", fontsize=11)
ax_cv.set_ylabel("Probability Density", fontsize=11)
ax_cv.set_title("Relative Channel Dispersion Comparison (CV)", fontsize=12, fontweight="bold", pad=6)
ax_cv.set_xlim(0, 30)
ax_cv.grid(True, linestyle=":", alpha=0.5)
ax_cv.legend(loc="upper right", fontsize=9.5, framealpha=0.92)

fig.suptitle("Unified Time-Stability: Per-Channel Variance & Dispersion Distributions", fontsize=15, fontweight="bold", y=1.005)
plt.tight_layout()
plt.show()


---
## 2. Exporting Single Unified Calibration per SiPM Channel

With time-stability demonstrated:
1. **Unified Shape Parameters:** Export $\hat{lpha}_i, \hat{eta}_i, \hat{Q}_{0,i}, \hat{\sigma}_{\mathrm{sys},i}$ with their pooled measurement uncertainties $\sigma_{\mathrm{pool}}$.
2. **Era-Specific Timing Alignments:** Export $\hat{d}_i^{(2022)}, \hat{d}_i^{(2023)}, \hat{d}_i^{(2024)}, \hat{d}_i^{(2025)}, \hat{d}_i^{(2026)}$ capturing the discrete LHC clock offsets.


In [ ]:
# Build final unified calibration dictionary
unified_dict = {
    "metadata": {
        "description": "Unified SND@LHC Time-Walk Calibration (Iteration 2)",
        "n_channels": len(df_stab_global),
        "stability_metrics": {
            "alpha_pooled_std": float(np.sqrt(np.mean(df_stab_global["alpha_var"].dropna()))),
            "beta_pooled_std": float(np.sqrt(np.mean(df_stab_global["beta_var"].dropna()))),
            "qdc0_pooled_std": float(np.sqrt(np.mean(df_stab_global["qdc0_var"].dropna()))),
            "sigma_sys_pooled_ps": float(np.sqrt(np.mean(df_stab_global["sigma_sys_var"].dropna())) * 1000.0),
            "d_sipm_intra_year_ps": float(np.sqrt(np.mean(df_stab_2022["d_sipm_var"].dropna())) * 1000.0)
        }
    },
    "timewalk": {},
    "alignment": {}
}

# Aggregate period-specific d_sipm medians
d_periods = df_good.groupby(["channel", "period"])["d_sipm"].median().unstack()

for _, row in df_stab_global.iterrows():
    ch = row["channel"]
    unified_dict["timewalk"][ch] = {
        "alpha": float(row["alpha_med"]),
        "alpha_err": float(row["alpha_std"]),
        "beta": float(row["beta_med"]),
        "beta_err": float(row["beta_std"]),
        "qdc0": float(row["qdc0_med"]),
        "qdc0_err": float(row["qdc0_std"]),
        "sigma_sys": float(row["sigma_sys_med"]),
        "sigma_sys_err": float(row["sigma_sys_std"]),
        "n_runs": int(row["n_runs"])
    }

periods_map = {
    "Cat II (2022)": "2022",
    "Cat III (2023)": "2023",
    "Cat III (2024)": "2024",
    "Cat III (2025)": "2025",
    "Cat III (2026)": "2026",
}

for p_orig, p_key in periods_map.items():
    unified_dict["alignment"][p_key] = {}
    if p_orig in d_periods.columns:
        s = d_periods[p_orig].dropna()
        for ch, val in s.items():
            unified_dict["alignment"][p_key][ch] = [float(val), 0.035]

out_json = "/eos/user/i/idioniso/Timing/timewalk_tw/unified_timing_calibration.json"
try:
    with open(out_json, "w") as f:
        json.dump(unified_dict, f, indent=2)
    print(f"Successfully exported consolidated calibration to {out_json}")
except Exception as e:
    print(f"Note: EOS export skipped or failed ({e}). File can be saved locally.")


---
## 3. Intra-Category Parameter Variance Comparison Across All 5 Eras

Directly evaluates and compares the run-to-run parameter repeatability within each individual operational period:
- **Cat II (2022)**
- **Cat III (2023)**
- **Cat III (2024)**
- **Cat III (2025)**
- **Cat III (2026)**


In [ ]:
# ==============================================================================
# Intra-Category Variance & Dispersion Comparison Across All 5 Eras
# ==============================================================================

periods_list = [
    "Cat II (2022)",
    "Cat III (2023)",
    "Cat III (2024)",
    "Cat III (2025)",
    "Cat III (2026)"
]

period_colors = {
    "Cat II (2022)": "#1f77b4",
    "Cat III (2023)": "#ff7f0e",
    "Cat III (2024)": "#2ca02c",
    "Cat III (2025)": "#d62728",
    "Cat III (2026)": "#9467bd"
}

# 1. Compute channel-level stability separately for each era
stab_by_era = {}
summary_era_rows = []

for p in periods_list:
    df_p = df_good[df_good["period"] == p]
    stab_p = compute_channel_stability(df_p, min_runs=5)
    stab_by_era[p] = stab_p
    
    # Calculate pooled root-mean-variance across channels in this era
    d_var = np.nanmean(stab_p["d_sipm_var"])
    a_var = np.nanmean(stab_p["alpha_var"])
    b_var = np.nanmean(stab_p["beta_var"])
    q_var = np.nanmean(stab_p["qdc0_var"])
    s_var = np.nanmean(stab_p["sigma_sys_var"])
    
    summary_era_rows.append({
        "Era / Category": p,
        "Channels Evaluated": len(stab_p),
        "d_SiPM sigma_pool [ps]": f"{np.sqrt(d_var)*1000:.1f} ps (med: {np.nanmedian(stab_p['d_sipm_std'])*1000:.1f})",
        "alpha sigma_pool": f"{np.sqrt(a_var):.2f} (med: {np.nanmedian(stab_p['alpha_std']):.2f})",
        "beta sigma_pool": f"{np.sqrt(b_var):.3f} (med: {np.nanmedian(stab_p['beta_std']):.3f})",
        "Q0 sigma_pool [QDC]": f"{np.sqrt(q_var):.3f} (med: {np.nanmedian(stab_p['qdc0_std']):.3f})",
        "sigma_sys pool [ps]": f"{np.sqrt(s_var)*1000:.1f} ps (med: {np.nanmedian(stab_p['sigma_sys_std'])*1000:.1f})"
    })

print("=== Per-Category Intra-Era Repeatability (Pooled Std & Median Channel Std) ===")
print(pd.DataFrame(summary_era_rows).to_string(index=False))

# 2. Multi-panel distribution of channel standard deviations by category
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes_flat = axes.flatten()

plot_params = [
    ("d_sipm",    r"Fine Time Alignment ($d_{\mathrm{SiPM}}$)",   "ps",  1000.0, (0, 300)),
    ("alpha",     r"Curvature Scale ($\alpha$)",                  "",    1.0,    (0, 10)),
    ("beta",      r"Scale Factor ($\beta$)",                     "",    1.0,    (0, 0.8)),
    ("qdc0",      r"Threshold Offset ($q_0$)",                   "QDC", 1.0,    (0, 1.5)),
    ("sigma_sys", r"Systematic Error Floor ($\sigma_{\mathrm{sys}}$)", "ps", 1000.0, (0, 30)),
]

for ax, (p_col, title, unit, scale, xlim) in zip(axes_flat[:5], plot_params):
    bins = np.linspace(xlim[0], xlim[1], 45)
    
    for p in periods_list:
        stab_p = stab_by_era[p]
        vals = stab_p[f"{p_col}_std"].dropna().values * scale
        med_val = np.median(vals)
        col = period_colors[p]
        ax.hist(vals, bins=bins, density=True, histtype="step", linewidth=1.8,
                color=col, label=f"{p} (med={med_val:.1f})")
    
    unit_str = f" [{unit}]" if unit else ""
    ax.set_xlabel(r"Per-Channel Standard Deviation $\hat{\sigma}_i$", fontsize=11)
    ax.set_ylabel("Probability Density", fontsize=11)
    ax.set_title(title, fontsize=12, fontweight="bold", pad=6)
    ax.set_xlim(xlim)
    ax.grid(True, linestyle=":", alpha=0.5)
    ax.legend(loc="upper right", fontsize=8.5, framealpha=0.92)

# Panel 6: Boxplot comparing per-category dispersion across eras for d_SiPM
ax_box = axes_flat[5]
box_data_d = [stab_by_era[p]["d_sipm_std"].dropna().values * 1000.0 for p in periods_list]
bp = ax_box.boxplot(
    box_data_d, labels=[p.replace("Cat ", "").replace("III ", "") for p in periods_list],
    patch_artist=True, showmeans=True,
    meanprops={"marker": "o", "markerfacecolor": "white", "markeredgecolor": "black", "markersize": 5},
    flierprops={"marker": ".", "markersize": 2, "alpha": 0.25}
)
for patch, p in zip(bp["boxes"], periods_list):
    patch.set_facecolor(period_colors[p])
    patch.set_alpha(0.65)

ax_box.set_ylabel(r"Intra-Era $\sigma(d_{\mathrm{SiPM}})$ per Channel [ps]", fontsize=11)
ax_box.set_title(r"Run-to-Run $d_{\mathrm{SiPM}}$ Stability by Era", fontsize=12, fontweight="bold", pad=6)
ax_box.set_ylim(0, 350)
ax_box.grid(True, linestyle=":", alpha=0.5)

fig.suptitle("Intra-Category Parameter Dispersion Comparison Across Eras", fontsize=15, fontweight="bold", y=1.005)
plt.tight_layout()
plt.show()


---
## 4. Testing Intra-Year Time Dependency Within a Single Category/Year

To test whether parameters are constant or drifting with time within the same operational period, we perform three rigorous tests on **Cat II (2022)** (33 consecutive runs across the full calendar year):
1. **Split-Half Epoch Shift (Early vs. Late):** Divides the year into the first half (runs 5119–5159) and second half (runs 5170–5408). Computes $\Delta p_i = p_i(	ext{Late}) - p_i(	ext{Early})$ for each channel.
2. **Linear Drift Slope ($):** Regresses (r) = m_i \cdot r + c_i$ against run number $ and computes total drift $\Delta p = m_i \cdot \Delta r$.
3. **Null-Hypothesis Significance Test (hBctest):** Tests whether slope $ is statistically distinguishable from zero ($|t_i| < 2.0$).


In [ ]:
# ==============================================================================
# Testing Intra-Year Time Dependency (Run-Number Drift Within the Same Year)
# ==============================================================================

# Focus on Cat II (2022) which has the longest continuous baseline (33 runs, 5119 to 5408)
df_2022 = df_good[df_good["period"] == "Cat II (2022)"].copy()

# Runs in 2022 sorted chronologically
runs_2022 = sorted(df_2022["run"].unique())
r_min, r_max = runs_2022[0], runs_2022[-1]
delta_r = r_max - r_min

# Split into Early 2022 (first 16 runs) vs Late 2022 (last 17 runs)
half_idx = len(runs_2022) // 2
early_runs = runs_2022[:half_idx]
late_runs = runs_2022[half_idx:]

df_early = df_2022[df_2022["run"].isin(early_runs)]
df_late = df_2022[df_2022["run"].isin(late_runs)]

# Calculate median per channel for early vs late
med_early = df_early.groupby("channel")[["d_sipm", "alpha", "beta", "qdc0"]].median()
med_late = df_late.groupby("channel")[["d_sipm", "alpha", "beta", "qdc0"]].median()
diff_intra = med_late - med_early

# Also calculate per-channel linear regression slope m_i vs run number
records_drift_intra = []
for ch, group in df_2022.groupby("channel"):
    if len(group) < 10:
        continue
    r_arr = group["run"].values
    
    rec = {"channel": ch, "n_runs": len(group)}
    for p in ["d_sipm", "alpha", "beta", "qdc0"]:
        vals = group[p].values
        # Fit straight line p(r) = m * r + c
        res = stats.linregress(r_arr, vals)
        slope = res.slope
        slope_err = res.stderr if res.stderr is not None and res.stderr > 0 else 1e-9
        t_stat = slope / slope_err
        total_drift = slope * delta_r
        
        rec[f"{p}_slope"] = slope
        rec[f"{p}_t_stat"] = t_stat
        rec[f"{p}_drift"] = total_drift
    records_drift_intra.append(rec)

df_drift_intra = pd.DataFrame(records_drift_intra)

print(f"=== Intra-Year Time-Dependency Test: Cat II 2022 ({len(runs_2022)} runs, run {r_min} to {r_max}) ===")
print(f"Channels evaluated: {len(df_drift_intra):,}
")

for p, name, unit, scale in [
    ("d_sipm", "d_SiPM", "ps", 1000.0),
    ("alpha",  "alpha ", "  ", 1.0),
    ("beta",   "beta  ", "  ", 1.0),
    ("qdc0",   "qdc0  ", "QDC", 1.0)
]:
    d_vals = diff_intra[p].dropna() * scale
    t_vals = df_drift_intra[f"{p}_t_stat"].dropna()
    drift_vals = df_drift_intra[f"{p}_drift"].dropna() * scale
    f_stat_stable = (np.abs(t_vals) < 2.0).mean() * 100.0
    
    print(f"{name:8s}: Early-to-Late Shift = {np.median(d_vals):+.2f} {unit} (RMS: {np.std(d_vals):.2f}) | "
          f"Total Linear Drift across 2022 = {np.median(drift_vals):+.2f} {unit} | "
          f"Consistent with 0 drift (|t| < 2): {f_stat_stable:.1f}%")

# Visual Diagnostic Figure
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

# 1. Early vs Late Shift in d_SiPM
ax1.hist(diff_intra["d_sipm"].dropna() * 1000.0, bins=np.linspace(-150, 150, 60), color="#1f77b4", edgecolor="black")
ax1.axvline(0, color="red", linestyle="--", linewidth=1.5, label="Zero Shift Baseline")
ax1.axvline(np.median(diff_intra["d_sipm"].dropna() * 1000.0), color="black", linestyle=":", linewidth=1.8,
            label=f"Median Shift: {np.median(diff_intra['d_sipm'].dropna() * 1000.0):+.1f} ps")
ax1.set_xlabel(r"$d_{\mathrm{SiPM}}^{\mathrm{Late}} - d_{\mathrm{SiPM}}^{\mathrm{Early}}$ [ps]", fontsize=11)
ax1.set_ylabel("Number of Channels", fontsize=11)
ax1.set_title(r"Intra-Year Shift: Fine Alignment ($d_{\mathrm{SiPM}}$)", fontweight="bold")
ax1.grid(True, linestyle=":", alpha=0.5)
ax1.legend(loc="upper right", fontsize=10)

# 2. Total Linear Drift across 2022 for d_SiPM
ax2.hist(df_drift_intra["d_sipm_drift"].dropna() * 1000.0, bins=np.linspace(-200, 200, 60), color="#ff7f0e", edgecolor="black")
ax2.axvline(0, color="red", linestyle="--", linewidth=1.5, label="Zero Drift Baseline")
ax2.axvline(np.median(df_drift_intra["d_sipm_drift"].dropna() * 1000.0), color="black", linestyle=":", linewidth=1.8,
            label=f"Median Drift: {np.median(df_drift_intra['d_sipm_drift'].dropna() * 1000.0):+.1f} ps")
ax2.set_xlabel(r"Total Extrapolated Drift $m_d \cdot \Delta r$ [ps]", fontsize=11)
ax2.set_ylabel("Number of Channels", fontsize=11)
ax2.set_title(r"Intra-Year Total Extrapolated Drift: $d_{\mathrm{SiPM}}$", fontweight="bold")
ax2.grid(True, linestyle=":", alpha=0.5)
ax2.legend(loc="upper right", fontsize=10)

# 3. Student t-statistic on Slope for alpha
ax3.hist(df_drift_intra["alpha_t_stat"].dropna(), bins=np.linspace(-4, 4, 60), color="#2ca02c", edgecolor="black")
ax3.axvspan(-2.0, 2.0, color="green", alpha=0.15, label="Null Hypothesis Retained (|t| < 2)")
ax3.axvline(-2.0, color="green", linestyle="--")
ax3.axvline(2.0, color="green", linestyle="--")
ax3.set_xlabel(r"Student's $t$-statistic on Linear Slope ($m_\alpha / \sigma_{m_\alpha}$)", fontsize=11)
ax3.set_ylabel("Number of Channels", fontsize=11)
ax3.set_title(r"Temporal Drift Significance Test: $\alpha$", fontweight="bold")
ax3.grid(True, linestyle=":", alpha=0.5)
ax3.legend(loc="upper right", fontsize=10)

# 4. Student t-statistic on Slope for d_SiPM
ax4.hist(df_drift_intra["d_sipm_t_stat"].dropna(), bins=np.linspace(-4, 4, 60), color="#d62728", edgecolor="black")
ax4.axvspan(-2.0, 2.0, color="green", alpha=0.15, label="Null Hypothesis Retained (|t| < 2)")
ax4.axvline(-2.0, color="green", linestyle="--")
ax4.axvline(2.0, color="green", linestyle="--")
ax4.set_xlabel(r"Student's $t$-statistic on Linear Slope ($m_d / \sigma_{m_d}$)", fontsize=11)
ax4.set_ylabel("Number of Channels", fontsize=11)
ax4.set_title(r"Temporal Drift Significance Test: $d_{\mathrm{SiPM}}$", fontweight="bold")
ax4.grid(True, linestyle=":", alpha=0.5)
ax4.legend(loc="upper right", fontsize=10)

fig.suptitle("Testing Intra-Year Parameter Drift Within Cat II (2022)", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()
